Volatility Modelling

Here we model volatility of S&P closing data using the GARCH framework. We start by converting closing prices into returns using logs. This makes return shocks become additive.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import minimize
import yfinance as yf
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf

data = yf.download(["^GSPC"], start="2005-01-01", end="2015-01-15")
closing_prices = data['Close'] 
print(closing_prices.head())

plt.plot(closing_prices, color='teal')
plt.title('S&P 500 Closing Prices (2005-2015)')
plt.grid(True)
plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.show()

returns = np.log(closing_prices).diff().fillna(0)
print(returns.head())
returns

Fetching S&P Data, convert to log returns. Additive over time. Visualise variance over time:

In [ ]:
np.var(returns) 
plt.plot(returns, color='teal')
plt.title('S&P 500 Log Returns (2005-2015)')
plt.grid(True)
plt.xlabel('Date')
plt.ylabel('Log Return')
plt.show()


Assume our return series has constant unconditional mean of mu. Model our return series as:

$$
r_{t} = \mu + \epsilon_{t}
$$

In [ ]:
ret = returns.iloc[:, 0].dropna() #First column, no time index
mu = ret.mean()

epsilon = ret - mu

plot_acf(epsilon, lags=40);

Unsurprisingly, no linear predictability in residuals. Support for A1 assumption, for Weak Form EMH, no linear predictability. However we can investigate predictability in higher moments, consider ACF of squared residuals:

In [ ]:
plot_acf(epsilon**2, lags=40);

Clear persistence. Evidence for conditional heteroscedasticity. Check if mu is statistically significant from 0.

In [ ]:
plt.hist(ret, bins=30)

se = ret.std(ddof=1) / np.sqrt(len(ret))
t_stat = mu / se
print(mu, t_stat)

if abs(t_stat) < 1.96:
    print("Treat mean as zero")
else:
    print("Keep nonzero mean")


At the 5% level, we can conclude that the mean is not statistically different from 0. Epsilon approximately equal to returns.

Our GARCH process accounts for conditional heteroscedasticity. Decompose our residual at time t:

$$
\epsilon_t = \sigma_tz_t, \:\:\:\:\:z_t \sim IID(0, 1)
$$

This residual follows a martingale difference process, such that:

$$
E[\epsilon_{t+1} |\Psi_t] = E[\epsilon_{t} |\Psi_t] = E[\epsilon_{t}] = 0 
$$
$$
E[\epsilon_{t} |\Psi_t] = E[\sigma_{t}z_{t}|\Psi_t] = E[\sigma_{t}|\Psi_t]E[z_t|\Psi_t] = \sigma_{t} \cdot0 = 0
$$

That is, our best guess of next period's innovation given the information up to time t is 0. One way martingale differences processes are represented, ARCH/GARCH residual decomposition. Variance of innovation predictable, however guess of next periods innovation is conditionally mean independent on the information set up to t. However volatility can be forecasted. Additionally:

$$
E[z^2_t] = 1
$$
$$
E[\epsilon^2_{t+1}|\Psi_{t}] = E[\sigma^2_{t+1}|\Psi_{t}]E[z^2_{t+1}|\Psi_{t}] = \sigma^2_t
$$

Squared zt term is 1 in expectation, variance of zt process as they are mean 0. So decomposing our residual and conditioning both components yields variance at t, as our best guess of next periods variance with this periods information is the same as this period.

Define our conditional variance:

$$
E[\sigma^2_{t+1}|\Psi_{t}] = E[\omega + \alpha\epsilon^2_{t} + \beta\sigma^2_{t}|\Psi_{t}] = \omega + \alpha\epsilon^2_{t} + \beta\sigma^2_{t}
$$

This is the equation for a standard GARCH(1,1) process. That is, our variance at time t is conditional on our previous squared residual along with an autoregressive term. Our unconditional long-run moment gives us:

$$
\bar{\sigma}^2 = \frac{\omega}{1 - \alpha - \beta}
$$

Define GARCH(1,1) Model:

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import minimize

def garch(omega, alpha, beta, epsilon): 
    
    length = len(epsilon)
    
    var = []

    for i in range(length):
        
        if i==0:
            var.append(omega/np.abs(1-alpha-beta)) # Unconditional variance
        else:
            var.append(omega + alpha * epsilon[i-1]**2 + beta * var[i-1]) # All points past 0 have lagged influences and new shocks
            
    return np.array(var)

Define MLE, we will use to fit our GARCH model:

In [ ]:
def likelihood(params, epsilon):
    
    length = len(epsilon)
    omega = params[0]
    alpha = params[1]
    beta = params[2]
    
    variance = garch(omega, alpha, beta, epsilon)
    
    llh = []

    for i in range(length):

        llh.append(np.log(norm.pdf(epsilon[i], 0, np.sqrt(variance[i]))))
    
    return -np.sum(np.array(llh))

Maximising log-likelihood, sum of log contributions for each data point. Find params which maximise (log) likelihood of observing the data.

In [ ]:
init = ((np.var(epsilon), 0.03, 0.85))
res = minimize(likelihood, init, epsilon,
               method = "Nelder-Mead")


In [ ]:
omega = res['x'][0] 
alpha = res['x'][1]
beta = res['x'][2]


condvar = garch(res['x'][0],res['x'][1],res['x'][2],epsilon)

plt.plot(condvar)